In [ ]:
import json
import pandas as pd
import sys
import time
from datetime import datetime


sys.path.append("jupyter_dir")
import uq_direction_semantic as uq

# -----------------------------
with open("data/qa/web-train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

qa_entries = data["Data"]

qa_list = [
    {
        "question": item["Question"],
        "answer_main": item["Answer"]["Value"],
        "answer_aliases": item["Answer"].get("Aliases", [])
    }
    for item in qa_entries
]

df_trivia = pd.DataFrame(qa_list)


In [ ]:
df_trivia.head()

In [ ]:
def my_prompt(row):
    if "context" in row and isinstance(row["context"], str) and row["context"].strip():
        return f"Context: {row['context']}\nQuestion: {row['question']}\nAnswer:"
    return f"Question: {row['question']}\nAnswer:"


df_fixed = df_trivia.sample(n=300, random_state=42).reset_index(drop=True)
model_ids = [
    "meta-llama/Llama-2-7b-chat-hf",
    "tiiuae/falcon-7b-instruct",
    "mistralai/Mistral-7B-Instruct-v0.2"
]

cache_dir = "model_cache"
hf_token  = "..."


outs = []

for mid in model_ids:
    safe_name = mid.replace("/", "_")
    out_csv = f"trivia300_{safe_name}.csv"

    print("\n" + "="*90)
    print(f"START MODEL: {mid}")
    print(f"Output CSV : {out_csv}")
    print(f"Start time : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    t0 = time.time()

    df_out = uq.run_uq(
        df_fixed,
        model_id=mid,
        prompt_fn=my_prompt,
        out_csv=out_csv,
        cache_dir=cache_dir,
        hf_token=hf_token,
        # generation settings (keep identical for fairness)
        n_generations=20,
        batch_size=2,
        max_new_tokens=64,
        temperature=0.7,
        top_p=0.9,
        # entropy baseline
        compute_semantic_entropy=True,
        entail_threshold=0.8,
    )

    t1 = time.time()
    print(f"END MODEL  : {mid}")
    print(f"End time  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Elapsed   : {(t1 - t0)/60:.2f} minutes")
    print("="*90)

    outs.append(df_out)

# -----------------------------
# 7) Combine results across models
# -----------------------------
df_all = pd.concat(outs, ignore_index=True)
out_all_csv = "trivia300_all3models.csv"
df_all.to_csv(out_all_csv, index=False)
print("\nSaved combined:", out_all_csv)
print("Combined shape:", df_all.shape)

In [ ]:
import numpy as np
from rouge_score import rouge_scorer
from sklearn.metrics import roc_auc_score

# --- use EXACT same scorer ---
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# --- EXACT same correctness function, only change is ref column name ---
def is_correct_rouge(row, threshold=0.1):
    generations = row["generations"]
    ref = row["answer_main"]   # TriviaQA reference column

    # Handle missing data
    if not isinstance(ref, str) or not isinstance(generations, list) or len(generations) == 0:
        return False

    top1 = generations[0]
    if not isinstance(top1, str):
        return False

    score = scorer.score(ref.lower(), top1.lower())
    rouge_l_f1 = score['rougeL'].fmeasure
    return rouge_l_f1 >= threshold


# df_all should be your combined TriviaQA results with 3 models
# Example: df_all = pd.read_csv("trivia300_all3models.csv")
# Make sure generations is a list already (in-memory df) OR parsed if loaded from CSV.

THRESH = 0.1

for model_name, df_results in df_all.groupby("model_id"):
    df_results = df_results.copy()

    # --- your exact pipeline ---
    df_results["is_correct"] = df_results.apply(lambda r: is_correct_rouge(r, threshold=THRESH), axis=1)
    df_results["y_true"] = (~df_results["is_correct"]).astype(int)  # 1 = incorrect, 0 = correct

    y_true = df_results["y_true"]

    # AUROC needs both classes
    if y_true.nunique() < 2:
        print(f"\nModel: {model_name}")
        print("Only one class present in y_true, AUROC undefined.")
        print("is_correct mean:", df_results["is_correct"].mean())
        continue

    auroc_kappa = roc_auc_score(y_true, 1 / df_results["kappa"])
    auroc_deberta = roc_auc_score(y_true, df_results["semantic_entropy"])

    print(f"\nModel: {model_name}")
    print(f"AUROC (1/kappa):         {auroc_kappa:.3f}")
    print(f"AUROC (deberta_entropy): {auroc_deberta:.3f}")


In [ ]:
import ast
import pandas as pd
import numpy as np

df_all = pd.read_csv("trivia300_all3models.csv")

def parse_generations(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if x == "" or x.lower() == "nan":
            return []
        try:
            return ast.literal_eval(x)   # parse "['a','b']" to ['a','b']
        except (SyntaxError, ValueError):
            # fallback: treat as single generation
            return [x]
    return []

df_all["generations"] = df_all["generations"].apply(parse_generations)


In [ ]:
import numpy as np
from rouge_score import rouge_scorer
from sklearn.metrics import roc_auc_score

# --- use EXACT same scorer ---
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# --- EXACT same correctness function, only change is ref column name ---
def is_correct_rouge(row, threshold=0.1):
    generations = row["generations"]
    ref = row["answer_main"]   # TriviaQA reference column

    # Handle missing data
    if not isinstance(ref, str) or not isinstance(generations, list) or len(generations) == 0:
        return False

    top1 = generations[0]
    if not isinstance(top1, str):
        return False

    score = scorer.score(ref.lower(), top1.lower())
    rouge_l_f1 = score['rougeL'].fmeasure
    return rouge_l_f1 >= threshold


# df_all should be your combined TriviaQA results with 3 models
# Example: df_all = pd.read_csv("trivia300_all3models.csv")
# Make sure generations is a list already (in-memory df) OR parsed if loaded from CSV.

THRESH = 0.1

for model_name, df_results in df_all.groupby("model_id"):
    df_results = df_results.copy()

    # --- your exact pipeline ---
    df_results["is_correct"] = df_results.apply(lambda r: is_correct_rouge(r, threshold=THRESH), axis=1)
    df_results["y_true"] = (~df_results["is_correct"]).astype(int)  # 1 = incorrect, 0 = correct

    y_true = df_results["y_true"]

    # AUROC needs both classes
    if y_true.nunique() < 2:
        print(f"\nModel: {model_name}")
        print("Only one class present in y_true, AUROC undefined.")
        print("is_correct mean:", df_results["is_correct"].mean())
        continue

    auroc_kappa = roc_auc_score(y_true, 1 / df_results["kappa"])
    auroc_deberta = roc_auc_score(y_true, df_results["semantic_entropy"])

    print(f"\nModel: {model_name}")
    print(f"AUROC (1/kappa):         {auroc_kappa:.3f}")
    print(f"AUROC (deberta_entropy): {auroc_deberta:.3f}")


In [ ]:
def my_prompt(row, max_ctx_chars=1500):
    """
    Prompt for SQuAD-style QA.
    Uses context when available and truncates to avoid overlong inputs.
    """
    context = row.get("context", None)
    question = row.get("question", "")

    if isinstance(context, str) and context.strip():
        context = context[:max_ctx_chars]
        return (
            "Answer the question using the passage below.\n\n"
            f"Passage: {context}\n\n"
            f"Question: {question}\n"
            "Answer:"
        )

    return f"Question: {question}\nAnswer:"

df_fixed = df_trivia.sample(n=30, random_state=40).reset_index(drop=True)

model_ids = [
    "meta-llama/Llama-2-7b-chat-hf",
    "tiiuae/falcon-7b-instruct",
    "mistralai/Mistral-7B-Instruct-v0.2"
]

gen_counts = [10, 20, 30, 40]   # <--- new
cache_dir = "model_cache"
hf_token  = "..."

outs = []

for mid in model_ids:
    for g in gen_counts:
        safe_name = mid.replace("/", "_")
        out_csv = f"squad300_{safe_name}_gen{g}.csv"

        print("\n" + "="*90)
        print(f"START MODEL: {mid} | n_gen={g}")
        print(f"Output CSV : {out_csv}")
        print(f"Start time : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        t0 = time.time()

        df_out = uq.run_uq(
            df_fixed,
            model_id=mid,
            prompt_fn=my_prompt,
            out_csv=out_csv,
            cache_dir=cache_dir,
            hf_token=hf_token,
            # --- variable part ---
            n_generations=g,
            # --- fixed settings ---
            batch_size=2,
            max_new_tokens=64,
            temperature=0.7,
            top_p=0.9,
            compute_semantic_entropy=True,
            entail_threshold=0.8,
        )

        t1 = time.time()
        print(f"END MODEL  : {mid} | n_gen={g}")
        print(f"End time   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Elapsed    : {(t1 - t0)/60:.2f} minutes")
        print("="*90)

        df_out["n_generations"] = g  # <--- add metadata column
        outs.append(df_out)

# Combine all results
df_all = pd.concat(outs, ignore_index=True)
out_all_csv = "trivia300_all3models_gen10_20_30_40.csv"
df_all.to_csv(out_all_csv, index=False)

print("\nSaved combined:", out_all_csv)
print("Combined shape:", df_all.shape)


In [ ]:
###Bootstrap
def compute_aurocs(df, threshold=0.1):
    rows = []
    for model_name, df_model in df.groupby("model_id"):
        df_model = df_model.copy()

        # compute correctness
        df_model["is_correct"] = df_model.apply(
            lambda r: is_correct_rouge(r, threshold=threshold),
            axis=1
        )
        df_model["y_true"] = (~df_model["is_correct"]).astype(int)  # 1=incorrect

        y_true = df_model["y_true"]

        # skip if only one class
        if y_true.nunique() < 2:
            rows.append({
                "model_id": model_name,
                "auroc_kappa": np.nan,
                "auroc_semantic": np.nan
            })
            continue

        auroc_kappa = roc_auc_score(y_true, 1 / df_model["kappa"])
        auroc_semantic = roc_auc_score(y_true, df_model["semantic_entropy"])

        rows.append({
            "model_id": model_name,
            "auroc_kappa": auroc_kappa,
            "auroc_semantic": auroc_semantic
        })

    return rows


In [ ]:
B = 1000
results = []

N = len(df_all)

for b in range(B):
    sample_idx = np.random.choice(N, size=N, replace=True)
    df_boot = df_all.iloc[sample_idx]

    rows = compute_aurocs(df_boot)
    for r in rows:
        r["bootstrap_id"] = b
        results.append(r)

df_bootstrap = pd.DataFrame(results)
df_bootstrap.to_csv("trivia300_auroc_all3models.csv", index = False)

In [ ]:
summary = (
    df_bootstrap
    .groupby("model_id")
    .agg(
        mean_kappa=("auroc_kappa", "mean"),
        std_kappa=("auroc_kappa", "std"),
        ci95_kappa=("auroc_kappa", lambda x: 1.96 * x.std()),

        mean_sem=("auroc_semantic", "mean"),
        std_sem=("auroc_semantic", "std"),
        ci95_sem=("auroc_semantic", lambda x: 1.96 * x.std()),
    )
)


In [ ]:
summary